In [1]:
import json
import os
import dashscope
from dashscope.api_entities.dashscope_response import Role
dashscope.api_key = "sk-你的KEY"
from openai import OpenAI

# 封装模型响应函数
def get_response(messages):
    response = dashscope.MultiModalConversation.call(
        model='qwen-vl-plus',
        messages=messages
    )
    return response

content = [
    {'image': 'https://aiwucai.oss-cn-huhehaote.aliyuncs.com/pdf_table.jpg'}, # Either a local path or an url
    {'text': '这是一个表格图片，帮我提取里面的内容，输出JSON格式'}
]

messages=[{"role": "user", "content": content}]
# 得到响应
response = get_response(messages)
response

MultiModalConversationResponse(status_code=<HTTPStatus.OK: 200>, request_id='48bffed2-7df7-9ac0-ac65-53508d1592cb', code='', message='', output=MultiModalConversationOutput(choices=[Choice(finish_reason='stop', message=Message({'role': 'assistant', 'content': [{'text': '好的，以下是整理后的生成 JSON 格式内容：\n\n```json\n{\n    "客户名称": "",\n    "联繫方式": "",\n    "产品型号": "",\n    "生产日期": "",\n    "数量": 0,\n    "使用年限": null,\n    "严重程度": "",\n    "急紧程度": "",\n    "问题点": [],\n    "退货": false,\n    "换货": false,\n    "维修": false,\n    "图例说明": "",\n    "描述人/提报人": {\n        "__DATE__": ""\n    },\n    "分析人": {\n        "__DATE__": ""\n    },\n    "原因归属": [\n        ""\n    ],\n    "零时措施": {},\n    "改善措施": {}\n}\n```\n\n请注意这个 JSON 对象中的键值对可能需要根据实际的表单结构进行调整。例如，“联系方 式”和“联系方式”的字段名应该是一致的；同样地，“严重程度”、“紧急程度”等也可能有误，请参照原图像自行修正。\n\n另外，在处理文本信息（如日期）的时候需要注意它们的具体形式，并在转换为 JSON 值之前正确解析这些数据。如果存在缺失或错误的数据项，则应相应地添加 `null` 或空字符串来表示该属性不存在或者没有提供具体的信息。'}]}))]), usage=MultiModalConversationUsage(input_tokens=1046, output_tokens=277))

In [2]:
print(response.output.choices[0].message.content[0]['text'])

好的，以下是整理后的生成 JSON 格式内容：

```json
{
    "客户名称": "",
    "联繫方式": "",
    "产品型号": "",
    "生产日期": "",
    "数量": 0,
    "使用年限": null,
    "严重程度": "",
    "急紧程度": "",
    "问题点": [],
    "退货": false,
    "换货": false,
    "维修": false,
    "图例说明": "",
    "描述人/提报人": {
        "__DATE__": ""
    },
    "分析人": {
        "__DATE__": ""
    },
    "原因归属": [
        ""
    ],
    "零时措施": {},
    "改善措施": {}
}
```

请注意这个 JSON 对象中的键值对可能需要根据实际的表单结构进行调整。例如，“联系方 式”和“联系方式”的字段名应该是一致的；同样地，“严重程度”、“紧急程度”等也可能有误，请参照原图像自行修正。

另外，在处理文本信息（如日期）的时候需要注意它们的具体形式，并在转换为 JSON 值之前正确解析这些数据。如果存在缺失或错误的数据项，则应相应地添加 `null` 或空字符串来表示该属性不存在或者没有提供具体的信息。


In [1]:
from openai import OpenAI
import os

api_key=os.getenv('MODELSCOPE_API_KEY')
print(api_key)

client = OpenAI(
    base_url='https://api-inference.modelscope.cn/v1',
    api_key=api_key, # ModelScope Token
)

ms-1f01126d-e6af-48a3-958b-7e927a4951ed


In [12]:
def get_message(content, isStream=True):
    response = client.chat.completions.create(
        model='stepfun-ai/Step-3.7-Flash',
        messages=content,
        stream=isStream
    )
    return response
def print_response(response):
    done_reasoning = False       # 标记思考是否结束
    is_reasoning_started = False # 标记思考是否开始
    
    for chunk in response:
        if chunk.choices:
            # 使用 getattr 安全取值，防止报错
            reasoning_chunk = getattr(chunk.choices[0].delta, 'reasoning_content', None) or ''
            answer_chunk = chunk.choices[0].delta.content or ''
            
            # 1. 处理思考过程
            if reasoning_chunk != '':
                # 如果是第一次收到思考内容，先打印 <think> 开头
                if not is_reasoning_started:
                    print('<think>', end='', flush=True)
                    is_reasoning_started = True
                # 打印思考片段
                print(reasoning_chunk, end='', flush=True)
                
            # 2. 处理最终回答
            elif answer_chunk != '':
                # 如果是第一次收到回答，说明思考结束了
                if not done_reasoning:
                    # 如果之前有思考过程，闭合 </think> 标签
                    if is_reasoning_started:
                        print('</think>\n\n', end='', flush=True)
                    done_reasoning = True
                    
                # 打印回答片段
                print(answer_chunk, end='', flush=True)


In [19]:
content = [
    {'image': 'https://aiwucai.oss-cn-huhehaote.aliyuncs.com/pdf_table.jpg'}, # Either a local path or an url
    {'text': '这是一个表格图片，帮我提取里面的内容，输出JSON格式'}
]

messages=[{"role": "user", "content": content}]

messages=[{
        'role':
            'user',
        'content': [{
            'type': 'text',
            'text': '这是一个表格图片，帮我提取里面的内容，输出JSON格式',
        }, {
            'type': 'image_url',
            'image_url': {
                'url':
                    # 'https://modelscope.oss-cn-beijing.aliyuncs.com/demo/images/audrey_hepburn.jpg',
                'https://aiwucai.oss-cn-huhehaote.aliyuncs.com/pdf_table.jpg',
            },
        }],
    }]
# 得到响应
response = get_message(messages)
for chunk in response:
    if chunk.choices:
        print(chunk.choices[0].delta.content, end='', flush=True)

```json
{
  "基本信息": {
    "客户名称": "",
    "客诉日期": "",
    "联系方式": "",
    "回复时间": "",
    "严重程度": {
      "一般": false,
      "重大": false
    },
    "紧急程度": {
      "一般": false,
      "紧急": false
    }
  },
  "产品信息": {
    "产品型号": "",
    "生产日期": {
      "年": "",
      "月": "",
      "日": ""
    },
    "数量": "1",
    "使用期限": "不详"
  },
  "客户诉求": {
    "问题产品追踪": {
      "客户处": false,
      "物流中": false,
      "已回厂": false
    },
    "客户诉求点": {
      "退货": false,
      "换货": false,
      "维修": false
    }
  },
  "描述区域": {
    "描述人/提报人": "",
    "日期": "2018年___月___日",
    "内容": "",
    "图例说明": ""
  },
  "问题要因分析": {
    "内容": "",
    "分析人": "",
    "日期": "2018年___月___日"
  },
  "临时措施": {
    "原因归属": {
      "设计": false,
      "可靠性": false,
      "品质部": false,
      "生产部": false,
      "仓储": false,
      "运输": false,
      "其它": false
    },
    "零时对策建议": "更换液压头，更换阀盘。",
    "对策方法": {
      "库存产品再检验": false,
      "退回二级供应商": false,
      "生产停产纠正": false,
      "其它": false
    }
  },
  "改善措施": {